In [32]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import joblib
import numpy as np

In [33]:
gender_map = {
    "Male": 0,
    "Female": 1
}
features = [
    "academic_performance",
    "academic_year",
    "age",
    "anxiety_score",
    "depression_score",
    "exam_pressure",
    "family_expectation",
    "financial_stress",
    "gender",
    "internet_usage",
    "mental_health_index",
    "physical_activity",
    "screen_time",
    "sleep_hours",
    "social_support",
    "stress_level",
    "study_hours_per_day"
]
predict_features = [
    "risk_level",
]

In [34]:
from pymongo import MongoClient
import pprint
client = MongoClient("mongodb://127.0.0.1:27017/?directConnection=true&serverSelectionTimeoutMS=2000&appName=mongosh+2.8.2")
db = client["student_burnout"]
collection = db["studentDetails"]


In [35]:
data = []
targets =[]
for doc in collection.find({}, {field: 1 for field in (features+ predict_features)}):
    row1 = []
    row2=[]
    for field in features:
        value = doc.get(field)
        if field == "gender":
            value = gender_map.get(value, -1)  # unknown = -1
        row1.append(value)
    for field in predict_features:
        value = doc.get(field)
        row2.append(value)
    data.append(row1)
    targets.append(row2)
X = np.array(data)
y = np.array(targets)

In [36]:
model = DecisionTreeRegressor()
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.2)
model.fit(X_train,y_train)
joblib.dump(model, "final-model.joblib")
predictions = model.predict(X_test)
print(mean_absolute_error(y_test, predictions))

0.194975
